# 1 — Sensor Data Simulation and Exploratory Analysis

**Sensor Intelligence Platform** — analytical walkthrough (1 / 7)

Real predictive-maintenance work rarely starts with clean, labelled fault data. This platform ships a **reproducible multivariate simulator** and a reusable reference asset — an eight-channel motor-driven pump skid — so every downstream stage (feature engineering, forecasting, anomaly detection, drift monitoring) can be developed and benchmarked against a signal whose ground truth we control.

In this notebook we:

1. Generate the full eight-sensor fleet with trend, daily seasonality, autocorrelated noise, and injected faults.
2. Inspect the raw series and their fault labels.
3. Decompose the structure: seasonal profile, noise distribution, autocorrelation.
4. Examine cross-sensor correlation and summarise the injected anomalies the rest of the platform must recover.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

NAVY, ORANGE, TEAL, RED, GREY = "#1f3a5f", "#e8893b", "#2a9d8f", "#c0392b", "#9aa0ad"

## 1.1 Generate the reference fleet

`default_fleet()` returns the generative specs for eight channels — `temperature` (C), `vibration` (g), `pressure` (kPa), `humidity` (%RH), `flow_rate` (m3/h), `motor_current` (A), `supply_voltage` (V), and `shaft_speed` (rpm). Each `SensorSpec` carries a baseline level, a linear `trend_per_day`, a `daily_amplitude` for the 24-hour cycle, and an AR(1) noise process. We sample at a 15-minute cadence for seven days and inject two faults: a transient **spike** on temperature and a **drift** ramp on vibration.

In [2]:
from sensor_intelligence.simulation import (
    SensorSimulator, SimulationConfig, AnomalyInjection, default_fleet,
)

STEP = 900           # 15-minute cadence
PERIOD = 96          # samples per day (24h * 4)
fleet = default_fleet()
config = SimulationConfig(
    sensors=fleet,
    n_steps=PERIOD * 7,
    step_seconds=STEP,
    seed=11,
    anomalies=[
        AnomalyInjection('temperature', start_step=PERIOD * 4 + 30, magnitude=18.0),
        AnomalyInjection('vibration', start_step=PERIOD * 5, duration=PERIOD,
                         magnitude=0.35, kind='drift'),
    ],
)
df = SensorSimulator(config).run()
print(f'{df.shape[0]} rows  |  {df.sensor_id.nunique()} sensors  |  '
      f'{df.timestamp.min():%Y-%m-%d} to {df.timestamp.max():%Y-%m-%d}')
df.head()

5376 rows  |  8 sensors  |  2024-01-01 to 2024-01-07


,sensor_id,timestamp,value,unit,is_anomaly
0,flow_rate,2024-01-01 00:00:00,122.782887,m3/h,False
1,flow_rate,2024-01-01 00:15:00,119.266583,m3/h,False
2,flow_rate,2024-01-01 00:30:00,121.963656,m3/h,False
3,flow_rate,2024-01-01 00:45:00,126.491726,m3/h,False
4,flow_rate,2024-01-01 01:00:00,127.178760,m3/h,False


The output is **tidy long-format** — one row per `(sensor_id, timestamp)` — exactly what an ingestion adapter would emit. The boolean `is_anomaly` column is the ground-truth label we will later score detectors against.

In [3]:
units = {s.sensor_id: s.unit for s in fleet}
order = [s.sensor_id for s in fleet]
summary = (df.groupby('sensor_id')
             .agg(n=('value', 'size'), mean=('value', 'mean'),
                  std=('value', 'std'), faults=('is_anomaly', 'sum'))
             .round(3)
             .loc[order])
summary['unit'] = summary.index.map(units)
summary

,n,mean,std,faults,unit
sensor_id,,,,,
temperature,672,62.570,5.025,1,C
vibration,672,0.444,0.079,95,g
pressure,672,101.660,1.119,0,kPa
humidity,672,44.363,8.611,0,%RH
flow_rate,672,120.103,13.034,0,m3/h
motor_current,672,14.171,1.575,0,A
supply_voltage,672,399.998,1.659,0,V
shaft_speed,672,1480.090,4.425,0,rpm


## 1.2 The raw series

We plot all eight channels on a shared time axis and highlight injected faults in red. The temperature spike is a single sharp excursion; the vibration drift is a slow ramp that a point detector will struggle with but a change-point or drift monitor will catch. The remaining channels carry only their natural trend, seasonality, and noise.

In [4]:
fig, axes = plt.subplots(4, 2, figsize=(13, 11), sharex=True)
for ax, sid in zip(axes.ravel(), order):
    s = df[df.sensor_id == sid]
    ax.plot(s.timestamp, s.value, color=NAVY, lw=0.8)
    faults = s[s.is_anomaly]
    if len(faults):
        ax.scatter(faults.timestamp, faults.value, color=RED, s=14, zorder=5,
                   label='injected fault')
        ax.legend(loc='upper left', fontsize=8)
    ax.set_title(f'{sid}  ({units[sid]})', fontsize=11)
fig.suptitle('Simulated pump-skid fleet with injected faults', y=0.995,
             fontsize=14, fontweight='bold')
for ax in axes[-1]:
    ax.set_xlabel('time')
fig.autofmt_xdate()
fig.tight_layout()

## 1.3 Seasonal profile

Averaging a channel by time-of-day recovers the imposed 24-hour cycle. This is why the feature layer adds **cyclical (sin/cos) calendar encodings** — they let a model see this structure as a smooth, continuous feature. Flow rate has the strongest daily swing in the fleet, so we profile it here.

In [5]:
def daily_profile(frame, sid):
    s = frame[(frame.sensor_id == sid) & (~frame.is_anomaly)].copy()
    s['minutes'] = s.timestamp.dt.hour * 60 + s.timestamp.dt.minute
    return s.groupby('minutes').value.agg(['mean', 'std'])

prof = daily_profile(df, 'flow_rate')
hours = prof.index / 60
fig, ax = plt.subplots()
ax.plot(hours, prof['mean'], color=NAVY, lw=2)
ax.fill_between(hours, prof['mean'] - prof['std'], prof['mean'] + prof['std'],
                color=NAVY, alpha=0.15, label='±1 std')
ax.set(title='Flow-rate daily profile (fault-free samples)',
       xlabel='hour of day', ylabel='flow_rate (m3/h)', xticks=range(0, 25, 3))
ax.legend(fontsize=9)
fig.tight_layout()

## 1.4 Noise structure

Subtracting the known trend-plus-seasonal component from a fault-free channel leaves the AR(1) noise. Its histogram is roughly Gaussian, and its autocorrelation decays geometrically at the rate set by `noise_ar` — exactly the structure the forecasting and anomaly layers must model rather than treat as independent.

In [6]:
temp = df[(df.sensor_id == 'temperature') & (~df.is_anomaly)].reset_index(drop=True)
days = (temp.timestamp - temp.timestamp.iloc[0]).dt.total_seconds().to_numpy() / 86_400.0
spec = next(s for s in fleet if s.sensor_id == 'temperature')
seasonal = (spec.baseline + spec.trend_per_day * days
            + spec.daily_amplitude * np.sin(2 * np.pi * days))
resid = temp.value.to_numpy() - seasonal

def acf(x, k):
    x = x - x.mean()
    denom = float(np.dot(x, x))
    return [float(np.dot(x[:x.size - i], x[i:]) / denom) for i in range(k)]

lags = acf(resid, 25)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.hist(resid, bins=40, color=NAVY, alpha=0.85)
a1.set(title='Temperature noise distribution', xlabel='residual (C)', ylabel='count')
a2.bar(range(len(lags)), lags, color=TEAL)
a2.axhline(0, color=GREY, lw=1)
a2.set(title='Residual autocorrelation (geometric decay)', xlabel='lag', ylabel='ACF')
fig.tight_layout()
print(f'lag-1 ACF = {lags[1]:.2f}  (spec.noise_ar = {spec.noise_ar})')

lag-1 ACF = 0.35  (spec.noise_ar = 0.4)


## 1.5 Cross-sensor correlation

Pivoting to wide format and correlating the fault-free samples exposes how the channels move together. In this simulator the coupling comes from the **shared daily cycle** — every channel with a daily amplitude rises and falls on the same 24-hour phase — which is exactly why de-seasonalising (or adding calendar features) before modelling matters: much of the raw cross-correlation is seasonality, not a genuine relationship between sensors.

In [7]:
wide = (df[~df.is_anomaly]
        .pivot_table(index='timestamp', columns='sensor_id', values='value')[order])
corr = wide.corr()

fig, ax = plt.subplots(figsize=(7.8, 6.6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
for i in range(len(order)):
    for j in range(len(order)):
        v = corr.iloc[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(v) > 0.55 else 'black')
ax.set_title('Cross-sensor correlation (fault-free samples)')
fig.colorbar(im, ax=ax, shrink=0.8, label='Pearson r')
fig.tight_layout()

## Takeaways

- The simulator emits **tidy, labelled, reproducible** multivariate data — a controlled ground truth for the whole platform.
- The reference **`default_fleet()`** gives every notebook the same eight channels with consistent units and scales.
- Each channel decomposes into **trend + daily seasonality + AR(1) noise**, with injected faults flagged for scoring.
- Raw cross-sensor correlation is dominated by **shared seasonality**, motivating the calendar features built in notebook 4.
- Next: engineer model-ready features ([notebook 4](04_feature_engineering.ipynb)) and forecast the series ([notebook 2](02_forecasting_and_backtesting.ipynb)).